# Experiment 3.2: Democracy Budget -- Partial Equalization

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/DEMOCRACY_BUDGET_partial_equalization/run_experiment.py`

This notebook is the analysis and visualization layer for the same toy experiment implemented in the script above. It intentionally **does not re-implement the optimizer or training loop**; instead, it imports the script, runs the same computation, and then presents the results in a more explicit, audit-friendly form.

## Scope and metric

The scientific scope here is deliberately narrow. This is a **48-parameter deep-linear toy problem** whose headline statistic is a **final-loss recovery metric**:

`recovery(method) = (L_SGD - L_method) / (L_SGD - L_Muon) * 100%`

Interpreted literally, this asks how much of Muon's final-loss advantage over SGD is recovered by full or partial Hessian-basis equalization. It is therefore a useful probe of this toy setup, but it is **not yet a direct mechanistic measurement** of oscillation damping, flat-direction acceleration, or basis alignment.


In [ ]:
import importlib.util
import json
import os
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass
plt.rcParams["figure.dpi"] = 120


def locate_repo_root():
    target = Path("experiments/Tier_1_Core_Mechanism_Tests/DEMOCRACY_BUDGET_partial_equalization/run_experiment.py")
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / target).exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory. "
        "Run the notebook from somewhere inside the Muon_as_RG_Gauge_Fixing repo."
    )


REPO_ROOT = locate_repo_root()
EXPERIMENT_DIR = REPO_ROOT / "experiments" / "Tier_1_Core_Mechanism_Tests" / "DEMOCRACY_BUDGET_partial_equalization"
SCRIPT_PATH = EXPERIMENT_DIR / "run_experiment.py"

spec = importlib.util.spec_from_file_location("democracy_budget_partial_equalization", SCRIPT_PATH)
exp = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = exp
spec.loader.exec_module(exp)

print(f"Repository root: {REPO_ROOT}")
print(f"Experiment directory: {EXPERIMENT_DIR}")
print(f"Imported script: {SCRIPT_PATH}")


## Reproducibility, runtime, and notebook execution policy

The default notebook run uses the **same default configuration as the script**. For smoke tests only, the code below honors the environment variable `PARTIAL_EQUALIZATION_NOTEBOOK_SMOKE=1` and switches to a much smaller configuration.

The runtime bottleneck is the finite-difference Hessian, which is recomputed repeatedly in a 48-parameter model. That is still manageable here, but this notebook should be read as a **toy-scope benchmark**, not as a scalable implementation.


In [ ]:
SMOKE_MODE = os.environ.get("PARTIAL_EQUALIZATION_NOTEBOOK_SMOKE", "0") == "1"
NOTEBOOK_CONFIG_OVERRIDES = {
    "n_seeds": 1,
    "n_steps": 50,
    "k_values": [1, 24],
    "lr_candidates": [0.001, 0.01],
    "hessian_recompute_every": 10,
} if SMOKE_MODE else {}
STORE_TRAJECTORIES = True

resolved_config = exp.prepare_config(NOTEBOOK_CONFIG_OVERRIDES)
planned_seeds = exp.make_seed_list(resolved_config)

run_log = {
    "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "matplotlib_version": plt.matplotlib.__version__,
    "smoke_mode": SMOKE_MODE,
    "store_trajectories": STORE_TRAJECTORIES,
    "script_path": str(SCRIPT_PATH),
    "planned_seeds": planned_seeds,
}

print("Run log:")
for key, value in run_log.items():
    print(f"  {key}: {value}")

print("\nResolved configuration:")
print(json.dumps(resolved_config, indent=2))


## Execute the counterpart script logic

The next cell runs the same experiment as the script and captures structured outputs for later tables and figures.


In [ ]:
start_time = time.time()
results = exp.run_experiment(
    config_overrides=NOTEBOOK_CONFIG_OVERRIDES,
    verbose=False,
    store_trajectories=STORE_TRAJECTORIES,
)
summary = exp.summarize_results(results)
notebook_runtime_seconds = time.time() - start_time

print(f"Notebook execution runtime: {notebook_runtime_seconds:.2f} seconds")
print(f"Script-reported runtime: {results['run_time_seconds']:.2f} seconds")
print(f"Seeds used: {results['seeds']}")
print(f"Representative partial k for trajectory view: {summary['representative_partial_k']}")


## Structured summaries

The following tables expose the resolved configuration, aggregate outcomes, and per-seed values that the figures later summarize.


In [ ]:
def display_table(rows, title=None, index=None):
    if title is not None:
        display(Markdown(f"### {title}"))
    if pd is not None:
        df = pd.DataFrame(rows)
        if index is not None and index in df.columns:
            df = df.set_index(index)
        display(df)
    else:
        for row in rows:
            print(row)
        print()


config_rows = [{"field": key, "value": value} for key, value in results["config"].items()]
run_rows = [
    {"field": "runtime_seconds_notebook", "value": notebook_runtime_seconds},
    {"field": "runtime_seconds_script", "value": results["run_time_seconds"]},
    {"field": "smoke_mode", "value": SMOKE_MODE},
    {"field": "store_trajectories", "value": results["store_trajectories"]},
    {"field": "seeds", "value": results["seeds"]},
]

display_table(run_rows, title="Run metadata")
display_table(config_rows, title="Resolved configuration")
display_table(summary["method_summary"], title="Aggregate method summary")
display_table(summary["recovery_by_k"], title="Recovery by implemented partial budget")
display_table(summary["per_seed_final_losses"], title="Per-seed final losses", index="seed")
display_table(summary["per_seed_best_lrs"], title="Per-seed best learning rates", index="seed")
display_table(summary["per_seed_recoveries"], title="Per-seed recoveries", index="seed")


## Recovery vs. k

This figure uses the **implemented** partial sweep only: `k ∈ {1, 2, 3, 5, 10, 15, 24}` by default. SGD is the conceptual no-equalization baseline, but there is **no implemented `k=0` partial-democratic run** in this experiment.


In [ ]:
k_values = results["config"]["k_values"]
seed_list = results["seeds"]
recovery_rows = summary["recovery_by_k"]

fig, ax = plt.subplots(figsize=(8, 5))

for seed_index, seed in enumerate(seed_list):
    seed_curve = [results["aggregate_arrays"]["PartialDemocratic_recoveries"][k][seed_index] for k in k_values]
    ax.plot(k_values, seed_curve, color="tab:blue", alpha=0.18, linewidth=1)
    ax.scatter(k_values, seed_curve, color="tab:blue", alpha=0.55, s=35)

mean_recoveries = [row["mean_recovery"] for row in recovery_rows]
sem_recoveries = [row["sem_recovery"] for row in recovery_rows]
ax.errorbar(
    k_values,
    mean_recoveries,
    yerr=sem_recoveries,
    color="black",
    marker="o",
    linewidth=2,
    capsize=4,
    label=f"Mean ± SEM (n={len(seed_list)})",
)
ax.axhline(100.0, color="tab:red", linestyle="--", linewidth=1.5, label="Muon parity = 100%")
ax.set_xlabel("k (bottom-k and top-k Hessian components equalized)")
ax.set_ylabel("Recovery relative to Muon (%)")
ax.set_title("Final-loss recovery versus partial equalization budget")
ax.legend(loc="best")
plt.show()


**Interpretation.** This is a budget curve for a toy **final-loss** statistic, not a direct shape analysis of the spectrum. The per-seed scatter shows how much variation is hidden by the mean, and the error bars make it explicit that any claim about the "minimum sufficient k" is currently mean-based and low-sample.

**Important caveat about `k=0`.** The point `k=0` is **conceptual only** in this experiment. SGD serves as that baseline, but the implemented partial-democratic sweep starts at `k=1` because the current operator always equalizes at least one low-curvature and one high-curvature eigendirection.


In [ ]:
method_rows = summary["method_summary"]
labels = [row["label"] for row in method_rows]
mean_losses = [row["mean_final_loss"] for row in method_rows]
sem_losses = [row["sem_final_loss"] for row in method_rows]
colors = []
for row in method_rows:
    if row["method"] == "SGD":
        colors.append("tab:gray")
    elif row["method"] == "Muon":
        colors.append("tab:orange")
    elif row["method"] == "DemocraticSGD_full":
        colors.append("tab:green")
    else:
        colors.append("tab:blue")

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(labels))
ax.bar(x, mean_losses, yerr=sem_losses, capsize=4, color=colors, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yscale("log")
ax.set_ylabel("Final loss (log scale; lower is better)")
ax.set_title("Mean final loss by method / partial budget")
plt.tight_layout()
plt.show()


**Interpretation.** The recovery metric is defined relative to SGD and Muon, but it is still valuable to inspect the underlying final losses directly. The log scale helps because the methods can differ by large factors. If two recovery values look similar while the loss bars remain noisy across seeds, the stronger claim should be the weaker one: this experiment suggests a pattern, but does not yet settle it.


In [ ]:
representative_k = summary["representative_partial_k"]
steps = np.arange(results["config"]["n_steps"])

trajectory_specs = [
    ("SGD", None, "SGD", "tab:gray"),
    ("Muon", None, "Muon", "tab:orange"),
    ("DemocraticSGD_full", None, "Democratic full", "tab:green"),
    ("PartialDemocratic", representative_k, f"Partial k={representative_k}", "tab:blue"),
]

fig, ax = plt.subplots(figsize=(9, 5))

for method_name, partial_k, label, color in trajectory_specs:
    stack = []
    for seed_result in results["seed_results"]:
        if method_name == "PartialDemocratic":
            trajectory = seed_result["best_run_trajectories"]["PartialDemocratic"][partial_k]
        else:
            trajectory = seed_result["best_run_trajectories"][method_name]
        stack.append(np.asarray(trajectory, dtype=float))
    stack = np.vstack(stack)

    for row in stack:
        ax.plot(steps, np.maximum(row, 1e-12), color=color, alpha=0.12, linewidth=1)
    ax.plot(
        steps,
        np.maximum(np.mean(stack, axis=0), 1e-12),
        color=color,
        linewidth=2.3,
        label=label,
    )

ax.set_yscale("log")
ax.set_xlabel("Optimization step")
ax.set_ylabel("Loss (log scale)")
ax.set_title("Best-LR trajectories from the same script run")
ax.legend(loc="best")
plt.show()


**Interpretation.** This trajectory view is deliberately minimal: it reuses the already-computed best-LR runs rather than introducing a second experiment. It can support statements about relative optimization speed or stability in this toy setup, but it still does **not** directly measure why a method behaves that way. For example, it does not isolate oscillation, flat-direction progress, or basis alignment.


## T1-T4 operational checks

These checks are retained from the script because they are useful summary diagnostics, but they should be read as **operational tests on current summary statistics**, not as definitive scientific tests.


In [ ]:
display_table(summary["hypothesis_tests"], title="Operational T1-T4 summary", index="test")


The most important caveat is that `T4` reports the **smallest implemented k whose mean recovery exceeds 100%**. That is not the same as a confidence-qualified threshold, nor does it prove a sharp phase transition in optimization behavior.


In [ ]:
full_recovery = summary["aggregate"]["democratic_full_mean_recovery"]
min_k_100 = summary["min_k_100_mean_recovery"]
representative_k = summary["representative_partial_k"]
n_seeds = summary["n_seeds"]
best_partial_k = summary["best_k_by_mean_recovery"]
best_partial_row = next(row for row in summary["recovery_by_k"] if row["k"] == best_partial_k)
best_partial_recovery = best_partial_row["mean_recovery"]

conclusion_md = f"""
## Calibrated conclusion

Under the current **final-loss recovery metric** in this deep-linear toy problem, **full Hessian-basis equalization** achieved a mean recovery of **{full_recovery:.1f}%** relative to Muon's gain over SGD.

- Smallest implemented `k` whose **mean** recovery exceeded 100%: **{min_k_100 if min_k_100 is not None else 'not reached'}**
- Best mean partial-democratic recovery in this run: **{best_partial_recovery:.1f}% at k = {best_partial_k}**
- Representative partial budget used in the trajectory figure: **k = {representative_k}**
- Number of seeds: **n = {n_seeds}**

### What this notebook currently supports

- It supports a transparent toy-scope readout of **per-seed final losses, per-seed recoveries, uncertainty-aware mean summaries, and one best-LR trajectory view from the same run**.
- It supports reading partial equalization as a **budget curve** in this benchmark: the mean recovery changes across implemented `k` values, and the present run's best partial point reached **{best_partial_recovery:.1f}%**.
- If larger `k` values perform better here, that is still evidence about this **toy final-loss probe**, not yet a direct mechanistic measurement.

### What this notebook does **not** establish

- It does **not** directly measure the mechanism behind the effect.
- It does **not** prove a robust spectral-shape law from a small number of mean points.
- It does **not** justify strong statistical language such as "robust threshold" from only `{n_seeds}` seeds.
- It does **not** show that the same pattern will persist beyond this deep-linear toy benchmark.

The correct reading of this first-pass notebook is therefore: **a more transparent and reproducible toy final-loss probe**, not a completed mechanistic study.
"""

display(Markdown(conclusion_md))
